In [1]:
from keras.preprocessing.text import Tokenizer

import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras import regularizers
from keras.callbacks import EarlyStopping

from keras.preprocessing import sequence
from tensorflow.keras.metrics import AUC
from keras import regularizers
import pandas as pd
import string
import tensorflow_addons as tfa
import numpy as np
from sklearn.model_selection import StratifiedKFold
import numpy as np
import itertools

In [2]:
def load_data():
    df = pd.read_csv("bert_dataset.csv") 
    
    X = df["text"].values
    y = df["generated"].values

    del df
    return X, y

In [3]:
def build_lstm_model(embed_dim, lstm_units, encoder):
    vocab_size = 5000
    maxlen = 128
    
    model = tf.keras.Sequential([
        encoder,
        tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=maxlen),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)),
        tf.keras.layers.Dropout(0.3),  
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),  
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    metrics = [
        tf.metrics.BinaryAccuracy(name='accuracy'),
        AUC(from_logits=False, name='auc'),
        tfa.metrics.F1Score(num_classes=1, threshold=0.5, name='f1', average='macro')
    ]
    
    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(1e-4),
        metrics=metrics
    )
    return model


In [ ]:
def run_grid_search():
    def tf_clean(text, label):
        text = tf.strings.regex_replace(text, rb"[^\x00-\x7F]+", b" ") 
        return text, label

    X, y = load_data()
    p = np.random.permutation(len(X))
    X = X[p]
    y = y[p]
    
    X = X[:20000]
    y = y[:20000]
    
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    param_grid = {
        "embed_dim": [128, 256, 512],
        "lstm_units": [64, 128, 256],
        "epochs": [5, 10, 15, 20]                          
    }

    results = []

    for embed_dim, lstm_units, epochs,  in itertools.product(
        param_grid["embed_dim"],
        param_grid["lstm_units"],
        param_grid["epochs"]
    ):
        fold_train_acc = []
        fold_train_auc = []
        fold_train_f1  = []
        fold_val_acc   = []
        fold_val_auc   = []
        fold_val_f1    = []

        print(f"\nTesting config: embed={embed_dim}, lstm={lstm_units}, epochs={epochs}")

        for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y)):
            X_train_raw = X[train_idx]
            y_train_raw = y[train_idx]
        
            X_test_raw = X[test_idx]
            y_test_raw = y[test_idx]

            train_ds = tf.data.Dataset.from_tensor_slices((X_train_raw, y_train_raw)).shuffle(len(X_train_raw)).batch(32)
            test_ds = tf.data.Dataset.from_tensor_slices((X_test_raw, y_test_raw)).batch(32)

            
            train_ds = train_ds.map(tf_clean)
            test_ds = test_ds.map(tf_clean)
            
            encoder = tf.keras.layers.TextVectorization(max_tokens=5000,output_sequence_length=128)
            encoder.adapt(train_ds.map(lambda text, label: text))
            model = build_lstm_model(
                embed_dim=embed_dim,
                lstm_units=lstm_units,
                encoder=encoder
            )

            history = model.fit(
                train_ds,
                validation_data=(test_ds),
                epochs=epochs,
                batch_size=32,
                verbose=0,
                callbacks=[
                    tf.keras.callbacks.EarlyStopping(
                        monitor='val_auc',
                        patience=5,        
                        restore_best_weights=True,
                        mode='max'
                    )
                ]
            )

            train_acc = history.history['accuracy'][-1]
            train_auc = history.history['auc'][-1]
            train_f1  = history.history['f1'][-1]
            
            val_acc = history.history['val_accuracy'][-1]
            val_auc = history.history['val_auc'][-1]
            val_f1  = history.history['val_f1'][-1]
            
            fold_train_acc.append(train_acc)
            fold_train_auc.append(train_auc)
            fold_train_f1.append(train_f1)
            
            fold_val_acc.append(val_acc)
            fold_val_auc.append(val_auc)
            fold_val_f1.append(val_f1)

            print(f"  Fold {fold+1} - train ACC={train_acc:.4f}, AUC={train_auc:.4f}, F1={train_f1:.4f}")
            print(f"  Fold {fold+1} - val ACC={val_acc:.4f}, AUC={val_auc:.4f}, F1={val_f1:.4f}")

        results.append({
            "embed_dim": embed_dim,
            "lstm_units": lstm_units,
            "epochs": epochs,
            "mean_train_acc": np.mean(fold_train_acc),
            "mean_train_auc": np.mean(fold_train_auc),
            "mean_train_f1": np.mean(fold_train_f1),
            "mean_val_acc": np.mean(fold_val_acc),
            "mean_val_auc": np.mean(fold_val_auc),
            "mean_val_f1": np.mean(fold_val_f1)
        })


    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by="mean_val_auc", ascending=False)

    return df_results

In [6]:
res = run_grid_search()


Testing config: embed=128, lstm=64, epochs=5
  Fold 1 - train ACC=0.9948, AUC=0.9994, F1=0.9948
  Fold 1 - val ACC=0.9650, AUC=0.9853, F1=0.9656
  Fold 2 - train ACC=0.9917, AUC=0.9979, F1=0.9917
  Fold 2 - val ACC=0.9735, AUC=0.9938, F1=0.9732
  Fold 3 - train ACC=0.9948, AUC=0.9995, F1=0.9947
  Fold 3 - val ACC=0.9657, AUC=0.9920, F1=0.9660
  Fold 4 - train ACC=0.9956, AUC=0.9993, F1=0.9956
  Fold 4 - val ACC=0.9800, AUC=0.9939, F1=0.9800
  Fold 5 - train ACC=0.9947, AUC=0.9992, F1=0.9947
  Fold 5 - val ACC=0.9712, AUC=0.9924, F1=0.9714

Testing config: embed=128, lstm=64, epochs=10
  Fold 1 - train ACC=0.9976, AUC=0.9997, F1=0.9976
  Fold 1 - val ACC=0.9720, AUC=0.9906, F1=0.9721
  Fold 2 - train ACC=0.9986, AUC=0.9997, F1=0.9986
  Fold 2 - val ACC=0.9790, AUC=0.9928, F1=0.9789
  Fold 3 - train ACC=0.9976, AUC=0.9996, F1=0.9976
  Fold 3 - val ACC=0.9755, AUC=0.9938, F1=0.9755
  Fold 4 - train ACC=0.9977, AUC=0.9998, F1=0.9977
  Fold 4 - val ACC=0.9755, AUC=0.9934, F1=0.9753
  Fold 

In [10]:
res.sort_values(by="mean_val_f1", ascending=False)

,embed_dim,lstm_units,epochs,mean_train_acc,mean_train_auc,mean_train_f1,mean_val_acc,mean_val_auc,mean_val_f1
2,128,64,15,0.997575,0.999683,0.997569,0.97695,0.992143,0.976891
29,512,128,10,0.996562,0.999798,0.996554,0.97675,0.992554,0.976703
21,256,256,10,0.994638,0.999466,0.994616,0.97625,0.994696,0.976134
15,256,64,20,0.996350,0.999574,0.996340,0.97610,0.992693,0.976029
3,128,64,20,0.996962,0.999732,0.996955,0.97585,0.991022,0.975828
24,512,64,5,0.994963,0.999420,0.994947,0.97590,0.993627,0.975806
35,512,256,20,0.996000,0.999736,0.995990,0.97565,0.991795,0.975553
25,512,64,10,0.996700,0.999746,0.996691,0.97530,0.991932,0.975383
31,512,128,20,0.995563,0.999654,0.995550,0.97550,0.991386,0.975379
4,128,128,5,0.993612,0.999307,0.993593,0.97530,0.994420,0.975207
